# 31 · EOPF: from STAC catalog to GeoZarr

**Time:** 35 minutes · **Audience:** undergraduate / introductory graduate GIS and Earth science.

**By the end:** trace an item to its arrays; distinguish resolution levels, storage shards and browser services.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-zarr-sql-views/lite/lab/index.html?path=31_eopf_discovery.ipynb) · [Earth lab](https://jltobias.github.io/JupyterLite-zarr-sql-views/lab/) · [Sources and licenses](https://jltobias.github.io/JupyterLite-zarr-sql-views/book/credits.html)

Run cells from top to bottom with **Shift+Enter**. The first kernel start downloads Python WebAssembly packages.
The bundled exercises need no data-service account. Save a copy with **File → Save and Export Notebook As** before clearing browser storage.

![Four-stage workflow: STAC discovery, Zarr reading, SQL selection, linked views](assets/pipeline.svg)

In [1]:
from course import *
import numpy as np
import matplotlib.pyplot as plt
print('Ready: Python + NumPy + local teaching data')

Ready: Python + NumPy + local teaching data


**Reproducibility record.** Project: Earth in the Browser; author: course contributors;
email/organization: not supplied. Created/updated: 2026-11-21. Input: `data/eopf/`;
outputs: the figures and objects below (exports are named explicitly). Python, NumPy and
Matplotlib; SQLite where used. Initial EOPF teaching release. The acquisition is a
convenience example, not a representative sample. Missing values remain missing.

![EOPF workflow from discovery to direct maps, quality-aware analysis and server processing](assets/eopf-workflow.svg)
## 1. Find the product before loading its pixels

Open the [Explorer](https://explorer.eopf.copernicus.eu/) and its
[STAC Browser](https://api.explorer.eopf.copernicus.eu/browser/). A catalog describes
products; it does not contain every pixel. This recorded item covers southern Italy.
The official tutorial's older item was absent when checked; this course pins a current
2026-09-28 item and keeps its metadata so class does not depend on catalog retention.

**Predict:** will the GeoZarr asset point to a single array, a group, or a file?

In [2]:
from eopf import load_eopf, search_url, live_json
d=load_eopf(); item=d['item']; p=d['provenance']
asset=item['assets']['reflectance']
print('Item:',item['id'])
print('Acquired:',p['date'],'| cloud cover (% of product):',item['properties']['eo:cloud_cover'])
print('Asset:',asset['href'])
print('Media type:',asset['type'],'| CRS:',asset['proj:code'])
print('Band names:',[b['name'] for b in asset['bands']])
print('License field:',d['collection']['license'])
print('Read the actual linked terms:',p['license_url'])

Item: S2C_MSIL2A_20260928T095031_N0513_R079_T33TVF_20260928T130916
Acquired: 2026-09-28T09:50:31.025000Z | cloud cover (% of product): 0.019449
Asset: https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/tests-output/sentinel-2-l2a/S2C_MSIL2A_20260928T095031_N0513_R079_T33TVF_20260928T130916.zarr/measurements/reflectance
Media type: application/vnd.zarr; version=3; profile=multiscales | CRS: EPSG:32633
Band names: ['b01', 'b02', 'b03', 'b04', 'b05', 'b06', 'b07', 'b08', 'b09', 'b11', 'b12', 'b8a']
License field: proprietary
Read the actual linked terms: https://sentinel.esa.int/documents/247904/690755/Sentinel_Data_Legal_Notice


## 2. Inspect groups, overview levels and codecs

The collection uses `proprietary` as its STAC license identifier and links the
Copernicus Sentinel legal notice. Do not relabel the data MIT or CC0 because a
viewer is open source. Preserve the linked terms and the modified-data attribution.

The source is **Zarr v3** with multiscales, projected coordinates and sharded arrays.
Our classroom store is a single-resolution, uncompressed GeoZarr v3 group. The older
SQL lab's Zarr v2 scalar schema is a different adapter; arbitrary source products
cannot be pasted into it unchanged.

In [3]:
specs=d['source-metadata']['consolidated_metadata']['metadata']
layout=specs['measurements/reflectance']['attributes']['multiscales']['layout']
for level in layout: print(level['asset'],level['spatial:shape'])
spec=specs['measurements/reflectance/r120m/b04']
print('Logical dtype:',spec['data_type'],'| source shard:',spec['chunk_grid'])
print('Inner codecs:',spec['codecs'][0]['configuration']['codecs'])
print('Source bytes retrieved:',p['source_bytes'],'| classroom pixels:',np.prod(p['shape']))

r10m [10980, 10980]
r20m [5490, 5490]
r60m [1830, 1830]
r120m [915, 915]
r360m [305, 305]
r720m [152, 152]
Logical dtype: float32 | source shard: {'name': 'regular', 'configuration': {'chunk_shape': [915, 915]}}
Inner codecs: [{'name': 'scale_offset', 'configuration': {'offset': -0.1, 'scale': 10000.0}}, {'name': 'cast_value', 'configuration': {'data_type': 'uint16', 'scalar_map': {'encode': [['NaN', 0]], 'decode': [[0, 'NaN']]}}}, {'name': 'bytes', 'configuration': {'endian': 'little'}}, {'name': 'blosc', 'configuration': {'typesize': 4, 'cname': 'zstd', 'clevel': 3, 'shuffle': 'shuffle', 'blocksize': 0}}]
Source bytes retrieved: 7227203 | classroom pixels: 16384


## 3. Search African cities without inventing coverage

The bounded Lagos and Cape Town searches returned zero items for the selected dates
on 2026-11-21. Catalog extent is not a guarantee of coverage. Try Abidjan, Lagos or
another collection/date interval and report the returned count before analysis.
The next cell defaults to URL construction; change `LIVE` to send one request.
A page limit is not the total matching count; follow `rel=next` deliberately if needed.

In [4]:
cities={'Lagos':[3.3,6.4,3.5,6.6],'Abidjan':[-4.1,5.2,-3.9,5.4],
        'Cape Town':[18.3,-34.1,18.6,-33.8]}
url=search_url(cities['Lagos'])
print(url)
LIVE=False
if LIVE:
    try:
        response=await live_json(url)
        print('Returned this page:',len(response['features']))
        print('Next links:',[x['href'] for x in response.get('links',[]) if x['rel']=='next'])
    except Exception as exc:
        print('LIVE REQUEST FAILED:',exc,'No fresh observations were obtained.')
else:
    print('RECORDED MODE: no live query was sent.')

https://api.explorer.eopf.copernicus.eu/stac/search?collections=sentinel-2-l2a&bbox=3.3%2C6.4%2C3.5%2C6.6&datetime=2026-08-01T00%3A00%3A00Z%2F2026-11-20T23%3A59%3A59Z&limit=5
RECORDED MODE: no live query was sent.


## Try it yourself

Inspect Sentinel-1 or Sentinel-3 metadata in STAC Browser. Explain which measurement, units and quality rules would change.

## Check your reasoning

Radar backscatter, optical reflectance and radiance need different processing. Shared Zarr storage does not make their physical meaning interchangeable.

## Evidence journal

Write four sentences: **what I observed; how I calculated it; what this cannot establish; what evidence I need next**.
For any figure you reuse, retain its units, date or scenario, data origin, transformations, and attribution.

## Sources and credit

- [EOPF Sentinel Zarr Explorer](https://explorer.eopf.copernicus.eu/), funded by ESA through the Copernicus Space Component programme; EOX, Development Seed and project contributors.
- [Copernicus Sentinel data legal notice](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice): contains modified Copernicus Sentinel data (2026). EOPF processing; course subset and repackaging.
- [Recorded source objects, checksums and transformations](https://jltobias.github.io/JupyterLite-zarr-sql-views/data/eopf/provenance.json). One 128 × 128 window of the upstream 120 m averaged overview, acquired 2026-09-28.

Teaching adaptation of [dzole0311/zarr-sql-views](https://github.com/dzole0311/zarr-sql-views), ISC.
Original teaching code and prose: JupyterLite-zarr-sql-views contributors, ISC. Data keep their separate licenses.

In [5]:
print('LESSON COMPLETE: 31_eopf_discovery')

LESSON COMPLETE: 31_eopf_discovery
